# Bench Tier A sim-disk plots

This notebook plots one raw `bench_tier_a/sim_disk` results directory. To plot a different run later, edit `RUN_DIR` in the first code cell or set `BENCH_TIER_A_RESULTS_DIR` before starting Jupyter.

`RUN_DIR` can be:

- A directory name under `pkg/kv/kvserver/bench_tier_a/sim_disk`, such as `raw-results-20260425-140725`.
- A path relative to the current notebook or repository root.
- An absolute path.

Included plots:

- Main effect: Raft log-commit latency by tier and scenario.
- Capture fractions, including signed regressions versus baseline.
- Latency percentile profiles.
- Client write histogram percentile curves from `*_hist.json`.
- Time-stability views from per-second histogram snapshots.
- WAL validation charts.

The ceiling sanity plot is intentionally omitted.

In [ ]:
import json
import math
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt

# Edit this for a different run, or set BENCH_TIER_A_RESULTS_DIR before
# starting Jupyter. Examples:
#   RUN_DIR = "raw-results-20260425-140725"
#   RUN_DIR = "../some-other-results"
#   RUN_DIR = "/absolute/path/to/raw-results-..."
RUN_DIR = os.environ.get("BENCH_TIER_A_RESULTS_DIR", "raw-results-20260425-140725")
SIM_DISK_REL = Path("pkg/kv/kvserver/bench_tier_a/sim_disk")


def resolve_results_dir(run_dir):
    """Resolve a raw results directory from common notebook launch locations."""
    requested = Path(run_dir).expanduser()
    if requested.is_absolute():
        candidates = [requested]
    else:
        candidates = []
        for parent in [Path.cwd(), *Path.cwd().parents]:
            candidates.append(parent / requested)
            candidates.append(parent / SIM_DISK_REL / requested)

    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if candidate.is_dir():
            return candidate

    tried = "\n  ".join(str(p) for p in candidates[:10])
    raise FileNotFoundError(
        f"Could not find RUN_DIR={run_dir!r}. Tried, for example:\n  {tried}\n"
        "Set RUN_DIR to an absolute path, a path relative to the notebook/repo, "
        "or export BENCH_TIER_A_RESULTS_DIR."
    )


RESULTS_DIR = resolve_results_dir(RUN_DIR)
RUN_NAME = RESULTS_DIR.name

DEFAULT_TIERS = ["nvme", "ssd-fast", "ssd-slow", "hdd"]
DEFAULT_SCENARIOS = ["baseline", "s1_p33", "s1_p100", "s2_p33", "s2_p100"]

# Keep the expected display order, but tolerate missing/extra tiers or scenarios
# in future raw-results directories with the same file naming convention.
TIERS = [tier for tier in DEFAULT_TIERS if (RESULTS_DIR / tier).is_dir()]
TIERS += sorted(
    tier.name for tier in RESULTS_DIR.iterdir()
    if tier.is_dir() and tier.name not in TIERS and any(tier.glob("pre_*.vars"))
)
if not TIERS:
    raise ValueError(f"No tier directories with pre_*.vars files found in {RESULTS_DIR}")

_discovered_scenarios = set()
for tier in TIERS:
    for path in (RESULTS_DIR / tier).glob("pre_*.vars"):
        scenario = path.name.removeprefix("pre_").removesuffix(".vars")
        if (RESULTS_DIR / tier / f"post_{scenario}.vars").exists():
            _discovered_scenarios.add(scenario)
SCENARIOS = [scenario for scenario in DEFAULT_SCENARIOS if scenario in _discovered_scenarios]
SCENARIOS += sorted(_discovered_scenarios.difference(SCENARIOS))
if not SCENARIOS:
    raise ValueError(f"No pre/post scenario .vars pairs found in {RESULTS_DIR}")

SCENARIO_LABELS = {
    "baseline": "baseline",
    "s1_p33": "D1 p=1/3",
    "s1_p100": "D1 p=1",
    "s2_p33": "D2 p=1/3",
    "s2_p100": "no fsync",
}


def scenario_label(scenario):
    return SCENARIO_LABELS.get(scenario, scenario)


PERCENTILES = [50, 75, 90, 95, 99, 99.9]

plt.rcParams.update({
    "figure.figsize": (12, 6),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

_BUCKET_RE = re.compile(r'^(\w+)_bucket\{.*?le="([^"]+)".*?\}\s+([\d.e+\-]+)')
_SCALAR_RE = re.compile(r'^(\w+)\{.*?\}\s+([\d.e+\-]+)')


def parse_vars_file(path):
    if not path.exists():
        return {}
    buckets = {}
    scalars = {}
    with path.open() as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            m = _BUCKET_RE.match(line)
            if m:
                name, le_s, val_s = m.groups()
                le = math.inf if le_s == "+Inf" else float(le_s)
                buckets.setdefault(name, {})[le] = buckets.setdefault(name, {}).get(le, 0.0) + float(val_s)
                continue
            m = _SCALAR_RE.match(line)
            if m:
                name, val_s = m.groups()
                scalars[name] = scalars.get(name, 0.0) + float(val_s)
    result = {name: sorted(vals.items()) for name, vals in buckets.items()}
    result.update(scalars)
    return result


def histogram_quantile(q, buckets):
    if not buckets:
        return float("nan")
    total = next((c for le, c in reversed(buckets) if le == math.inf), buckets[-1][1])
    if total <= 0:
        return 0.0
    target = q * total
    prev_le, prev_c = 0.0, 0.0
    for le, c in buckets:
        if le == math.inf:
            return prev_le
        if c >= target:
            if c == prev_c:
                return prev_le
            frac = (target - prev_c) / (c - prev_c)
            return prev_le + frac * (le - prev_le)
        prev_le, prev_c = le, c
    return float("nan")


def delta_buckets(pre, post):
    return [(le_post, max(0.0, c_post - c_pre)) for (_, c_pre), (le_post, c_post) in zip(pre or [], post or [])]


def ns_to_ms(v):
    return v / 1e6 if v == v and not math.isinf(v) else None


def round_or_none(v, n=3):
    if v is None or math.isnan(v) or math.isinf(v):
        return None
    return round(v, n)


print(f"Using results directory: {RESULTS_DIR}")
print(f"Run name: {RUN_NAME}")
print(f"Tiers: {', '.join(TIERS)}")
print(f"Scenarios: {', '.join(SCENARIOS)}")

In [ ]:
# HdrHistogram snapshot support for the client write histograms in *_hist.json.
# This is a small Python translation of the indexing logic from hdrhistogram-go.
class Hdr:
    def __init__(self, lowest, highest, sigfigs, counts):
        self.lowest = int(lowest)
        self.highest = int(highest)
        self.sigfigs = int(sigfigs)
        self.counts = [int(x) for x in counts]

        largest = 2 * (10 ** self.sigfigs)
        self.sub_bucket_count_magnitude = math.ceil(math.log2(largest))
        self.sub_bucket_half_count_magnitude = max(1, self.sub_bucket_count_magnitude) - 1
        self.unit_magnitude = max(0, math.floor(math.log2(self.lowest)))
        self.sub_bucket_count = 1 << (self.sub_bucket_half_count_magnitude + 1)
        self.sub_bucket_half_count = self.sub_bucket_count // 2
        self.sub_bucket_mask = (self.sub_bucket_count - 1) << self.unit_magnitude

        smallest_untrackable = self.sub_bucket_count << self.unit_magnitude
        buckets = 1
        while smallest_untrackable < self.highest:
            smallest_untrackable <<= 1
            buckets += 1
        self.bucket_count = buckets
        self.total = sum(self.counts)

    @classmethod
    def from_snapshot(cls, snap):
        return cls(
            snap["LowestTrackableValue"],
            snap["HighestTrackableValue"],
            snap["SignificantFigures"],
            snap["Counts"],
        )

    def value_from_index(self, bucket_idx, sub_bucket_idx):
        return int(sub_bucket_idx) << int(bucket_idx + self.unit_magnitude)

    def bucket_base_idx(self, bucket_idx):
        return (bucket_idx + 1) << self.sub_bucket_half_count_magnitude

    def count_at(self, bucket_idx, sub_bucket_idx):
        idx = self.bucket_base_idx(bucket_idx) + sub_bucket_idx - self.sub_bucket_half_count
        return self.counts[idx] if 0 <= idx < len(self.counts) else 0

    def bucket_index(self, v):
        return int(v | self.sub_bucket_mask).bit_length() - self.unit_magnitude - (self.sub_bucket_half_count_magnitude + 1)

    def sub_bucket_idx(self, v, bucket_idx):
        return int(v) >> int(bucket_idx + self.unit_magnitude)

    def size_equiv_range(self, v, bucket_idx):
        adjusted = bucket_idx + (1 if self.sub_bucket_idx(v, bucket_idx) >= self.sub_bucket_count else 0)
        return 1 << int(self.unit_magnitude + adjusted)

    def lowest_equiv(self, v, bucket_idx):
        return self.value_from_index(bucket_idx, self.sub_bucket_idx(v, bucket_idx))

    def highest_equiv(self, v):
        bucket = self.bucket_index(v)
        return self.lowest_equiv(v, bucket) + self.size_equiv_range(v, bucket) - 1

    def value_at_percentile(self, percentile):
        if self.total <= 0:
            return 0
        percentile = min(100.0, max(0.0, float(percentile)))
        target = int(((percentile / 100.0) * self.total) + 0.5)
        count_to_idx = 0
        bucket_idx = 0
        sub_bucket_idx = -1
        value_from_idx = 0
        while count_to_idx < target:
            sub_bucket_idx += 1
            if sub_bucket_idx >= self.sub_bucket_count:
                sub_bucket_idx = self.sub_bucket_half_count
                bucket_idx += 1
            if bucket_idx >= self.bucket_count:
                break
            count_to_idx += self.count_at(bucket_idx, sub_bucket_idx)
            value_from_idx = self.value_from_index(bucket_idx, sub_bucket_idx)
        if percentile == 0:
            return value_from_idx
        return self.highest_equiv(value_from_idx)

    def mean(self):
        if self.total <= 0:
            return 0.0
        total = 0.0
        bucket_idx = 0
        sub_bucket_idx = -1
        while True:
            sub_bucket_idx += 1
            if sub_bucket_idx >= self.sub_bucket_count:
                sub_bucket_idx = self.sub_bucket_half_count
                bucket_idx += 1
            if bucket_idx >= self.bucket_count:
                break
            count = self.count_at(bucket_idx, sub_bucket_idx)
            if count:
                value = self.value_from_index(bucket_idx, sub_bucket_idx)
                total += count * self.highest_equiv(value)
        return total / self.total


def read_hist_records(path):
    records = []
    if not path.exists():
        return records
    with path.open() as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            hist = Hdr.from_snapshot(obj["Hist"])
            records.append({
                "now": obj.get("Now"),
                "elapsed_ns": obj.get("Elapsed"),
                "total": hist.total,
                "p50": hist.value_at_percentile(50) / 1e6,
                "p95": hist.value_at_percentile(95) / 1e6,
                "p99": hist.value_at_percentile(99) / 1e6,
                "hist": hist,
            })
    return records


def aggregate_hist(records):
    if not records:
        return None
    base = records[0]["hist"]
    counts = [0] * len(base.counts)
    for record in records:
        for i, count in enumerate(record["hist"].counts):
            counts[i] += count
    return Hdr(base.lowest, base.highest, base.sigfigs, counts)

In [ ]:
def load_logcommit_metrics():
    metrics = {tier: {} for tier in TIERS}
    for tier in TIERS:
        for scenario in SCENARIOS:
            pre = parse_vars_file(RESULTS_DIR / tier / f"pre_{scenario}.vars")
            post = parse_vars_file(RESULTS_DIR / tier / f"post_{scenario}.vars")
            hist_name = "raft_process_logcommit_latency"

            count = max(0.0, post.get(f"{hist_name}_count", 0.0) - pre.get(f"{hist_name}_count", 0.0))
            total_ns = max(0.0, post.get(f"{hist_name}_sum", 0.0) - pre.get(f"{hist_name}_sum", 0.0))
            buckets = delta_buckets(pre.get(hist_name), post.get(hist_name)) if pre.get(hist_name) and post.get(hist_name) else []

            wal_bytes = max(0.0, post.get("storage_wal_bytes_written", 0.0) - pre.get("storage_wal_bytes_written", 0.0))
            fsyncs = max(0.0, post.get("storage_wal_fsync_latency_count", 0.0) - pre.get("storage_wal_fsync_latency_count", 0.0))

            metrics[tier][scenario] = {
                "avg": round_or_none(ns_to_ms(total_ns / count) if count else None),
                "p50": round_or_none(ns_to_ms(histogram_quantile(0.50, buckets))),
                "p95": round_or_none(ns_to_ms(histogram_quantile(0.95, buckets))),
                "p99": round_or_none(ns_to_ms(histogram_quantile(0.99, buckets))),
                "commits": int(count),
                "wal_gb": round(wal_bytes / 1e9, 3),
                "fsyncs": int(round(fsyncs)),
            }
    return metrics


def load_capture_fractions(metrics):
    captures = {}
    for tier in TIERS:
        m = metrics[tier]
        b = m["baseline"]["avg"]
        s1p = m["s1_p33"]["avg"]
        s1u = m["s1_p100"]["avg"]
        s2p = m["s2_p33"]["avg"]
        s2u = m["s2_p100"]["avg"]

        def pct(num, den):
            if den is None or abs(den) < 1e-12:
                return None
            return round(100.0 * num / den, 1)

        captures[tier] = {
            "d1": pct(b - s1p, b - s1u),
            "d2": pct(b - s2p, b - s2u),
            "write_bw": pct(s1p - s2p, b - s2u),
        }
    return captures


def load_client_histograms():
    distributions = {tier: {} for tier in TIERS}
    time_series = {tier: {} for tier in TIERS}
    for tier in TIERS:
        for scenario in SCENARIOS:
            records = read_hist_records(RESULTS_DIR / tier / f"{scenario}_hist.json")
            agg = aggregate_hist(records)
            distributions[tier][scenario] = {
                "percentiles": PERCENTILES,
                "values": [round((agg.value_at_percentile(p) / 1e6), 2) if agg else None for p in PERCENTILES],
                "mean": round((agg.mean() / 1e6), 2) if agg else None,
                "samples": int(agg.total if agg else 0),
            }
            time_series[tier][scenario] = records
    return distributions, time_series


metrics = load_logcommit_metrics()
captures = load_capture_fractions(metrics)
distributions, time_series = load_client_histograms()

print("Loaded:")
print(f"  {len(TIERS)} tiers")
print(f"  {len(SCENARIOS)} scenarios")
print(f"  {sum(len(time_series[t][s]) for t in TIERS for s in SCENARIOS)} histogram snapshots")

In [ ]:
# Numeric summary. If pandas is available, show a compact dataframe; otherwise print rows.
rows = []
for tier in TIERS:
    for scenario in SCENARIOS:
        row = {"tier": tier, "scenario": scenario, **metrics[tier][scenario]}
        rows.append(row)

try:
    import pandas as pd

    summary_df = pd.DataFrame(rows)
    display(summary_df)

    capture_df = pd.DataFrame([
        {"tier": tier, **captures[tier]} for tier in TIERS
    ])
    display(capture_df)
except ImportError:
    for row in rows:
        print(row)
    print("\nCapture fractions:")
    for tier in TIERS:
        print(tier, captures[tier])

## Main effect: Raft log-commit latency

Grouped bars compare `avg`, `p95`, and `p99` logcommit latency for each scenario within each disk tier. Values come from Prometheus histogram deltas between `pre_*.vars` and `post_*.vars`.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9), sharey=False)
for ax, tier in zip(axes.flat, TIERS):
    x = list(range(len(SCENARIOS)))
    width = 0.25
    for offset, key in [(-width, "avg"), (0, "p95"), (width, "p99")]:
        ax.bar([i + offset for i in x], [metrics[tier][s][key] for s in SCENARIOS], width=width, label=key)
    ax.set_title(tier)
    ax.set_xticks(x)
    ax.set_xticklabels([scenario_label(s) for s in SCENARIOS], rotation=25, ha="right")
    ax.set_ylabel("ms")
    ax.legend()
fig.suptitle(f"Raft log-commit latency by tier and scenario: {RUN_NAME}", y=1.02)
fig.tight_layout()
plt.show()

## Capture fractions

Positive values mean the scenario improved avg logcommit latency versus baseline. Negative values mean the scenario regressed versus baseline on this metric.

In [ ]:
capture_keys = [
    ("d1", "D1 capture @ p=1/3"),
    ("d2", "D2 capture @ p=1/3"),
    ("write_bw", "write-BW share"),
]

fig, ax = plt.subplots(figsize=(13, 6))
y = []
labels = []
values = []
colors = []
for tier in TIERS:
    for key, label in capture_keys:
        value = captures[tier][key]
        y.append(len(y))
        labels.append(f"{tier} · {label}")
        values.append(value)
        colors.append("tab:blue" if value >= 0 else "tab:gray")

ax.barh(y, values, color=colors)
ax.axvline(0, color="black", linewidth=1)
ax.set_yticks(y)
ax.set_yticklabels(labels)
ax.set_xlabel("% of tier-local ceiling")
ax.set_title("Signed capture fractions from avg logcommit latency")
for yi, value in zip(y, values):
    ax.text(value + (5 if value >= 0 else -5), yi, f"{value:.1f}%", va="center", ha="left" if value >= 0 else "right")
fig.tight_layout()
plt.show()

## Latency percentile profile

Each panel shows how the `p50`, `p95`, and `p99` logcommit percentiles move across scenarios for one tier.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9), sharey=False)
for ax, tier in zip(axes.flat, TIERS):
    x = list(range(len(SCENARIOS)))
    for key in ["p50", "p95", "p99"]:
        ax.plot(x, [metrics[tier][s][key] for s in SCENARIOS], marker="o", label=key)
    ax.set_title(tier)
    ax.set_xticks(x)
    ax.set_xticklabels([scenario_label(s) for s in SCENARIOS], rotation=25, ha="right")
    ax.set_ylabel("ms")
    ax.legend()
fig.suptitle(f"Raft log-commit percentile profile: {RUN_NAME}", y=1.02)
fig.tight_layout()
plt.show()

## Client write histogram percentile curves

These curves are computed by summing the per-second HdrHistogram snapshots in each `*_hist.json` file, then evaluating selected percentiles.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9), sharey=False)
percentile_labels = [f"p{p:g}" for p in PERCENTILES]
x = list(range(len(PERCENTILES)))
for ax, tier in zip(axes.flat, TIERS):
    for scenario in SCENARIOS:
        ax.plot(x, distributions[tier][scenario]["values"], marker="o", label=scenario_label(scenario))
    ax.set_title(tier)
    ax.set_xticks(x)
    ax.set_xticklabels(percentile_labels)
    ax.set_ylabel("client write latency (ms)")
    ax.legend(fontsize=8)
fig.suptitle(f"Client write latency percentile curves: {RUN_NAME}", y=1.02)
fig.tight_layout()
plt.show()

## Time stability

The next cell plots per-second client write latency percentiles for one tier. Change `TIME_TIER` to inspect another tier.

In [ ]:
TIME_TIER = "hdd"  # try any value from TIERS
if TIME_TIER not in TIERS:
    raise ValueError(f"TIME_TIER must be one of: {', '.join(TIERS)}")

fig, axes = plt.subplots(len(SCENARIOS), 1, figsize=(14, max(3 * len(SCENARIOS), 6)), sharex=False)
if len(SCENARIOS) == 1:
    axes = [axes]
for ax, scenario in zip(axes, SCENARIOS):
    records = time_series[TIME_TIER][scenario]
    x = list(range(len(records)))
    ax.plot(x, [r["p50"] for r in records], label="p50")
    ax.plot(x, [r["p95"] for r in records], label="p95")
    ax.plot(x, [r["p99"] for r in records], label="p99")
    ax.set_title(f"{TIME_TIER} · {scenario_label(scenario)}")
    ax.set_ylabel("ms")
    ax.legend(loc="upper right")
axes[-1].set_xlabel("seconds into run")
fig.suptitle(f"Per-second client write latency stability: {RUN_NAME} · {TIME_TIER}", y=1.01)
fig.tight_layout()
plt.show()

## WAL validation

These charts validate mechanism-level movement by plotting WAL bytes written and Pebble WAL fsync count deltas by scenario.

In [ ]:
fig, axes = plt.subplots(len(TIERS), 2, figsize=(15, max(3.5 * len(TIERS), 7)), sharex=False)
if len(TIERS) == 1:
    axes = [axes]
for row, tier in enumerate(TIERS):
    x = list(range(len(SCENARIOS)))

    ax = axes[row][0]
    ax.bar(x, [metrics[tier][s]["wal_gb"] for s in SCENARIOS])
    ax.set_title(f"{tier}: WAL bytes")
    ax.set_ylabel("GB")
    ax.set_xticks(x)
    ax.set_xticklabels([scenario_label(s) for s in SCENARIOS], rotation=25, ha="right")

    ax = axes[row][1]
    ax.bar(x, [metrics[tier][s]["fsyncs"] for s in SCENARIOS])
    ax.set_title(f"{tier}: WAL fsyncs")
    ax.set_ylabel("count")
    ax.set_xticks(x)
    ax.set_xticklabels([scenario_label(s) for s in SCENARIOS], rotation=25, ha="right")

fig.suptitle(f"WAL validation metrics: {RUN_NAME}", y=1.01)
fig.tight_layout()
plt.show()